<a href="https://colab.research.google.com/github/ReethikaDhulipalla2005/Sentiment-Analysis-using-BiLSTM/blob/main/Multi_Class_Sentiment_Analysis_using_BiLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

# plotting
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# -------------------------------------------------------------------
#                        CONFIGURATION
# -------------------------------------------------------------------
FILE_PATHS = ["https://colab.research.google.com/drive/1_-l6FCfmXh62EWbjSD-UjTi9Ekc3H0Il?usp=sharing"]


VOCAB_SIZE = 10000
MAX_LEN = 50
EMBEDDING_DIM = 128
LSTM_UNITS = 64
BATCH_SIZE = 64
EPOCHS = 10


In [ ]:
# -------------------------------------------------------------------
#                        CLEANING FUNCTION
# -------------------------------------------------------------------
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

In [ ]:
def load_sentiment140_data(filepath, is_manual=False):
    """Loads Sentiment140 format CSV."""
    if not os.path.exists(filepath):
        return pd.DataFrame()
    try:
        cols = ['sentiment_int', 'ids', 'date', 'flag', 'user', 'text']
        df = pd.read_csv(filepath, encoding='latin-1', header=None, names=cols)
        df = df[['text', 'sentiment_int']].copy()

        # mapping for manual: includes neutral
        if is_manual:
            sentiment_mapping = {0: 'negative', 2: 'neutral', 4: 'positive'}
        else:  # full dataset has only 0 and 4
            sentiment_mapping = {0: 'negative', 4: 'positive'}

        df['sentiment'] = df['sentiment_int'].map(sentiment_mapping)

        df.dropna(subset=['text', 'sentiment'], inplace=True)
        df['text'] = df['text'].astype(str).apply(clean_text)

        return df[['text', 'sentiment']]
    except Exception as e:
        print(f"Warning: failed to load {filepath}: {e}")
        return pd.DataFrame()

In [ ]:
# -------------------------------------------------------------------
#                     LOAD & COMBINE ALL DATA
# -------------------------------------------------------------------
df_list = [
    load_kaggle_data(FILE_PATHS['kaggle_train']),
    load_kaggle_data(FILE_PATHS['kaggle_test']),
    load_sentiment140_data(FILE_PATHS['sentiment140_manual'], is_manual=True),
    load_sentiment140_data(FILE_PATHS['sentiment140_full'], is_manual=False)
]

# remove empty
df_list = [d for d in df_list if not d.empty]

if not df_list:
    raise ValueError("No CSV files loaded. Check your paths.")

combined_df = pd.concat(df_list, ignore_index=True)
combined_df.drop_duplicates(subset=['text'], keep='first', inplace=True)

# keep only valid sentiments
valid_sentiments = ['negative', 'neutral', 'positive']
combined_df = combined_df[combined_df['sentiment'].isin(valid_sentiments)].copy()

if combined_df.empty:
    raise ValueError("No valid sentiment data available after filtering.")

# distribution
print("Class distribution:\n", combined_df['sentiment'].value_counts())

ValueError: No CSV files loaded. Check your paths.

In [ ]:
# -------------------------------------------------------------------
#                      LABEL ENCODING
# -------------------------------------------------------------------
X = combined_df['text'].values
y_str = combined_df['sentiment'].values

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_str)
class_names = list(label_encoder.classes_)

NUM_CLASSES = len(class_names)
print("\nLabel mapping:")
for i, c in enumerate(class_names):
    print(f"{c} → {i}")

# train–val split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
# -------------------------------------------------------------------
#                  TOKENIZING & PADDING
# -------------------------------------------------------------------
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<oov>")
tokenizer.fit_on_texts(X_train)

X_train_padded = pad_sequences(
    tokenizer.texts_to_sequences(X_train),
    maxlen=MAX_LEN, padding='post', truncating='post'
)
X_val_padded = pad_sequences(
    tokenizer.texts_to_sequences(X_val),
    maxlen=MAX_LEN, padding='post', truncating='post'
)

In [ ]:
# -------------------------------------------------------------------
#                  MODEL DEFINITION
# -------------------------------------------------------------------
model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(LSTM_UNITS, return_sequences=False)),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# -------------------------------------------------------------------
#                     TRAINING
# -------------------------------------------------------------------
early_stop = EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

history = model.fit(
    X_train_padded,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val_padded, y_val),
    callbacks=[early_stop],
    verbose=1
)
# -------------------------------------------------------------------
#                     EVALUATION
# -------------------------------------------------------------------
loss, acc = model.evaluate(X_val_padded, y_val, verbose=0)
print(f"\nValidation Accuracy: {acc:.4f}")

In [ ]:
# -------------------------------------------------------------------
#                     PLOTTING
# -------------------------------------------------------------------
plt.figure(figsize=(10, 4))

# accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history.get("accuracy", []))
plt.plot(history.history.get("val_accuracy", []))
plt.title("Accuracy")
plt.legend(["Train", "Validation"])
plt.grid(True)

# loss
plt.subplot(1, 2, 2)
plt.plot(history.history.get("loss", []))
plt.plot(history.history.get("val_loss", []))
plt.title("Loss")
plt.legend(["Train", "Validation"])
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# -------------------------------------------------------------------
#                 CONFUSION MATRIX & REPORT
# -------------------------------------------------------------------
y_pred = model.predict(X_val_padded, verbose=0)
y_pred_cls = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_val, y_pred_cls)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

print("\nClassification Report:\n")
print(classification_report(y_val, y_pred_cls, target_names=class_names))

In [ ]:
# -------------------------------------------------------------------
#            INTERACTIVE PREDICTOR
# -------------------------------------------------------------------
def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([clean_text(text)])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post")
    pred = model.predict(padded, verbose=0)
    return class_names[np.argmax(pred)]

# Run only if executed directly
if __name__ == "__main__":
    print("\n--- Interactive Predictor ---")
    while True:
        user_in = input("Enter text (or 'quit'): ")
        if user_in.lower() == "quit":
            break
        print("Sentiment:", predict_sentiment(user_in))